### Lesson 1.3 - Word Embeddings (Word2Vec)
---

<b>1.3.1: The Problem with TF-IDF</b><br>
TF-IDF vectors are sparse meaning majority of the entries are zero with no concrete meaning in them.

Example:
```js
"king" ka TF-IDF vector: [0.2, 0, 0, 0.1, 0, ...]
"queen" ka TF-IDF vector: [0, 0.15, 0, 0, 0.08, ...]
```

No such mathematical relation can be found in both the vectors, even though "king" and "queen" are both semantically related

Question: How can be create vectors that are close to each other for related words?<br>
-> Word embeddings! specifically, Word2Vec.

---

<b>1.3.2: Word2Vec - The Magic of Dense Vectors</b><br>

Core idea:

- Every word is represented as a **dense vector** (Ex: a 100-dimensional vector)
- The vectors for related words should be close to each other
- Example: `vector("king") - vector("man") + vector("woman") ≈ vector("queen")`

**How it works:**

Concept of Word2Vec is: "You shall know a word by the company it keeps"

If "apple" and "mango" are always represented with words like "tasty", "fruit", "juicy" then both the embeddings should be close to each other

---

<b>1.3.3: Two Flavors of Word2Vec</b><br>

1. CBOW (Continuous Bag of Words):

- Goal: Predicting target word by looking at the context word
- Example: "the quick ____ fox jumps" → predict "brown"
- Use case: Works exceptionally well on smaller datasets

2. Skip-gram:

- Goal: Predicting context word by looking at the target word
- Example: "brown" → predict "the", "quick", "fox", "jumps"
- Use case: Works well on bigger datasets, good for rare words

**We will focus on the Skip-gram architecture**

---

<b>1.3.4: Skip-gram Architecture (Math Derivation)</b><br>

Example:
```js
Sentence: "the quick brown fox jumps"
Target word: "brown"
Context window (size 1): "quick", "fox"
```
Step 1: Input Representation

- Convert the target word "brown" into a one-hot vector
- If vocab size is 5000, then this vector will be 5000 elements long
- Only the index for "brown" will be 1, others will be 0

Step 2: The hidden layer (The Magic Matrix)

- A weight matrix $W$ will have a shape of `(vocab_size, embedding_dim)`
- Example: `(5000, 100)`
- When the one-hot vector is multiplied with this matrix $W$ then the resulting vector will be a 100-dimendional dense vector
- CRITICAL INSIGHT: every row in $W$ is a embedding for a word

Step 3: The Output Layer

- Multiply this 100-dimensional vector with another Weight Matrix $W'$
- Shape of $W'$: (embedding_dim, vocab_size) -> (100, 5000)
- Result: 5000-dimensional vector (score of every word)
- Apple **Softmax** function -> We'll get the probability distribution

Step 4: The Loss Function

- Our goal: The probability of actual context words ("quick" and "fox") should be close to 1
- Loss function: **Negative Log-Likelihood (Cross-Entropy Loss)**
- Formula: `Loss = -log(P(context_word | target_word))`

Training:

- Train the model on all (Target, Context) pairs
- Calculate the gradients (Backpropagation)
- Update the weights (Gradient Descent)
- Repeat for multiple epochs

---

In [15]:
# 1. Data Preparation - Skip-gram pairs
import torch

corpus = ["i love machine learning", "machine learning is fun"]

vocab = {}
idx = 0

for sent in corpus:
    sent = sent.split()
    for word in sent:
        if word not in vocab:
            vocab[word] = idx
            idx += 1

print(vocab)

def gen_pair(sentences, vocab, window_size=1):
    pairs = []
    for sent in sentences:
        words = sent.split()
        for i, word in enumerate(words):
            for j in range(i - window_size, i + window_size + 1):
                if i == j:
                    continue
                if j >= 0 and j <= len(words) - 1:
                    context_word = words[j]
                    pairs.append((vocab[word], vocab[context_word]))

    return pairs

pairs = gen_pair(corpus, vocab, window_size=1)
print(f"Total pairs: {len(pairs)}")
print(f"Sample pairs: {pairs[:5]}")

targets = [pair[0] for pair in pairs]
context = [pair[1] for pair in pairs]

target_tensor = torch.tensor(targets, dtype=torch.long)
context_tensor = torch.tensor(context, dtype=torch.long)

print(f"Targets shape: {target_tensor.shape}")
print(f"Context shape: {context_tensor.shape}")

{'i': 0, 'love': 1, 'machine': 2, 'learning': 3, 'is': 4, 'fun': 5}
Total pairs: 12
Sample pairs: [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3)]
Targets shape: torch.Size([12])
Context shape: torch.Size([12])


In [16]:
# 2. The Skip-gram Model
import torch.nn as nn
from torch.nn import functional as F

class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, target_ids):
        embeds = self.embeddings(target_ids)
        logits = self.linear(embeds)
        log_probs = F.log_softmax(logits, dim=1)

        return log_probs

# testing the model
vocab_size = len(vocab)
embed_dim = 10

model = SkipGramModel(vocab_size, embed_dim)

dummpy_input = torch.tensor([0, 1, 2, 3])
output = model(dummpy_input)

print(f"Input shape: {dummpy_input.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([4])
Output shape: torch.Size([4, 6])


In [17]:
import torch.optim as optim

criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 100

for epoch in range(num_epochs):
    log_probs = model(target_tensor)
    loss = criterion(log_probs, context_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch: {epoch + 1}/{num_epochs} | Loss: {loss.item():.4f}")

embedding_matrix = model.embeddings.weight.data

machine_idx = vocab['machine']
learning_idx = vocab['learning']
machine_vec = embedding_matrix[machine_idx]
learning_vec = embedding_matrix[learning_idx]

def cosine_sim(vec1, vec2):
    numerator = torch.dot(vec1, vec2)
    denominator = torch.norm(vec1) * torch.norm(vec2)
    sim_score = numerator/denominator
    return sim_score

print(f"Similarity between 'machine' and 'learning': {cosine_sim(machine_vec, learning_vec):.4f}")

Epoch: 20/100 | Loss: 1.1099
Epoch: 40/100 | Loss: 0.7169
Epoch: 60/100 | Loss: 0.6020
Epoch: 80/100 | Loss: 0.5745
Epoch: 100/100 | Loss: 0.5650
Similarity between 'machine' and 'learning': -0.4429


### End of lesson 1.3

In this lesson, we did the following:
- Data preparation with Pairs generated with their respective Indices
- Defined a SkipGram Model architecture
- Trained the model on our micro dataset.
- Got the similarity score between the words 'machine' and 'learning'.

---